In [10]:
import json
import sys
sys.path.append(r"C:\Users\Abhinav Arora\Downloads")

file_path = r"C:\Users\Abhinav Arora\Desktop\COCO custom model\Original JSON files\coco_wholebody_train_v1.0.json"
with open(file_path, "r") as file:
    data = json.load(file)

In [11]:
#Editing thr categories field now
for c in data["categories"]:
    c["keypoints"] += ["left_big_toe", "right_big_toe", "left_heel", "right_heel"]
    c["skeleton"] += [[16,18], [16,20], [17,19], [17,21]]

In [12]:
#Reconfiguring the JSON file to remove unecessary data from the annotations:
keys_to_keep = set(["id", "image_id", "category_id", "segmentation", "area", "bbox", "iscrowd", "keypoints", "num_keypoints", "foot_valid"])

#Indices for left big toe in foot_kpts = 0, 1, 2
#Indices for right big toe in foot_kpts = 9, 10, 11
#Indices for left heel in foot_kpts = 6, 7, 8
#Indices for right heel in foot_kpts = 15, 16, 17

annotations = []
for a in data["annotations"]:
    copy_a = a.copy()
    # Zeroing out the foot coordinates in case they are garbage values and foot_valid is false
    if copy_a["foot_valid"] == False:
        copy_a["foot_kpts"] = [0] * 18
    
    left_big_toe_kpts = copy_a["foot_kpts"][0:3]
    right_big_toe_kpts = copy_a["foot_kpts"][9:12]
    left_heel_kpts = copy_a["foot_kpts"][6:9]
    right_heel_kpts = copy_a["foot_kpts"][15:18]
    copy_a["keypoints"] = copy_a["keypoints"] + left_big_toe_kpts + right_big_toe_kpts + left_heel_kpts + right_heel_kpts

    keys_to_remove = [key for key in copy_a if key not in keys_to_keep]
    for key in keys_to_remove:
        del copy_a[key]
    
    keypoint_count = 0
    #Going through all keypoints to evaluate their validity. It is always the third index
    for i in range(1, (len(copy_a["keypoints"])//3) + 1):
        ind = (i * 3) - 1
        if copy_a["keypoints"][ind] != 0:
            keypoint_count += 1
    
    copy_a["num_keypoints"] = keypoint_count
    annotations.append(copy_a)

In [13]:
#Reassigning annotations:
data["annotations"] = annotations

new_file_path = r"C:\Users\Abhinav Arora\Desktop\COCO custom model\dataset\annotations\training_data_COCO21.json"
with open(new_file_path, "w") as file:
    json.dump(data, file, indent=4)

keys in annotation:

segmentation
num_keypoints
area
iscrowd
keypoints
image_id
bbox
category_id
id
face_box
lefthand_box
righthand_box
lefthand_kpts
righthand_kpts
face_kpts
face_valid
lefthand_valid
righthand_valid
foot_valid
foot_kpts

following shows the revised keypoints:
{
    "id": int,                    # unique annotation id
    "image_id": int,              # links to an entry in "images"
    "category_id": int,           # 1 for person
    "segmentation": [...],        # polygon segmentation (can ignore)
    "area": float,                # bounding box area
    "bbox": [x, y, w, h],        # bounding box
    "iscrowd": 0,                 # 0 for individual, 1 for crowd
    "keypoints": [x1,y1,v1, x2,y2,v2, ... x17,y17,v17],  # 51 values flat + 6 values for each left and right big toe
    "num_keypoints": int          # number of labeled keypoints -> 19 total for the 2 toes as well (maximum possible)
}

In [14]:
#Now going through the images and filtering out the annotations with image ids that aren't in the folder
import os

def load_images_from_folder(folder):
    return set(os.listdir(folder))

images = load_images_from_folder(r"C:\Users\Abhinav Arora\Desktop\COCO custom model\dataset\coco_images")

with open(new_file_path, "r") as file:
    data = json.load(file)

#Removing the annotations that aren't in the images set
indices_to_remove = set()
index = 0
for a in data["annotations"]:
    image_id = a["image_id"]
    img_name = f"{str(image_id).zfill(12)}.jpg"
    if img_name not in images:
        indices_to_remove.add(index)
    index += 1

index = 0
new_annotations = []
for i in range(len(data["annotations"])):
    if i not in indices_to_remove:
        new_annotations.append(data["annotations"][i])

data["annotations"] = new_annotations

In [15]:
with open(new_file_path, "w") as file:
    json.dump(data, file, indent=4)